# 04 - FALCON-ID and RareGuard Embedding

Train the first-paper-style FALCON-ID proxy and the RareGuard rare-aware embedding model.


In [1]:
from pathlib import Path
import os, sys

LOCAL_CANDIDATES = [
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]

def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'rareguard').exists()

def find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents, *LOCAL_CANDIDATES, *DRIVE_CANDIDATES]:
        if is_project_root(candidate):
            return candidate
    if Path('/content').exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as exc:
            print('Google Drive mount skipped/failed:', exc)
        for candidate in DRIVE_CANDIDATES:
            if is_project_root(candidate):
                return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'RareGuard-XIDS IMPLEMENTATION project not found. '
        'If you are using Colab connector, Colab cannot read /Volumes/LocalDrive1 directly. '
        'Copy/sync the IMPLEMENTATION folder to Google Drive at '
        '/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION, '
        'or run this notebook with a local VS Code Python kernel on your Mac.'
    )
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)
print('Python path ready')
for name in ['logs', 'results', 'processed', 'features', 'models', 'figures', 'reports']:
    (PROJECT_ROOT / 'artifacts' / name).mkdir(parents=True, exist_ok=True)
print('Artifact folders ready:', PROJECT_ROOT / 'artifacts')


Project root: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION
Python path ready
Artifact folders ready: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION/artifacts


In [2]:
DATASET_KEY = 'all'
DEVICE = 'cpu'  # change to cuda only after GPU is attached and torch detects it
!python -m rareguard.experiments.run_embedding --config configs/default.yaml --datasets configs/datasets.yaml --dataset {DATASET_KEY} --device {DEVICE}


2026-09-25 05:13:42,301 | INFO | rareguard | cse_cic_ids2018 FalconID f1_macro=0.8075
2026-09-25 05:13:52,412 | INFO | rareguard | cse_cic_ids2018 RareGuard f1_macro=0.8632
2026-09-25 05:18:10,644 | INFO | rareguard | cic_bccc_nrc_tabulariotattack_2024 FalconID f1_macro=0.9975
2026-09-25 05:18:18,575 | INFO | rareguard | cic_bccc_nrc_tabulariotattack_2024 RareGuard f1_macro=0.9992
2026-09-25 05:23:27,700 | INFO | rareguard | cic_iiot_2025_datasense FalconID f1_macro=0.3721
2026-09-25 05:23:35,819 | INFO | rareguard | cic_iiot_2025_datasense RareGuard f1_macro=0.5637
2026-09-25 05:28:05,397 | INFO | rareguard | edge_iiotset FalconID f1_macro=0.9993
2026-09-25 05:28:11,597 | INFO | rareguard | edge_iiotset RareGuard f1_macro=0.9993
2026-09-25 05:28:11,619 | INFO | rareguard | Wrote artifacts/results/embedding_metrics.csv


In [3]:
import pandas as pd
metrics = pd.read_csv('artifacts/results/embedding_metrics.csv')
display(metrics.sort_values(['dataset', 'model']))


,dataset,model,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,model_path,vectors_path
2,cic_bccc_nrc_tabulariotattack_2024,FalconID,0.999211,0.998886,0.996180,0.997528,0.999210,artifacts/models/cic_bccc_nrc_tabulariotattack...,artifacts/models/cic_bccc_nrc_tabulariotattack...
3,cic_bccc_nrc_tabulariotattack_2024,RareGuard,0.999737,0.999856,0.998501,0.999177,0.999737,artifacts/models/cic_bccc_nrc_tabulariotattack...,artifacts/models/cic_bccc_nrc_tabulariotattack...
4,cic_iiot_2025_datasense,FalconID,0.849800,0.385135,0.391879,0.372104,0.827536,artifacts/models/cic_iiot_2025_datasense_Falco...,artifacts/models/cic_iiot_2025_datasense_Falco...
5,cic_iiot_2025_datasense,RareGuard,0.954100,0.595101,0.567352,0.563730,0.949769,artifacts/models/cic_iiot_2025_datasense_RareG...,artifacts/models/cic_iiot_2025_datasense_RareG...
0,cse_cic_ids2018,FalconID,0.985600,0.850662,0.785736,0.807549,0.982933,artifacts/models/cse_cic_ids2018_FalconID.pt,artifacts/models/cse_cic_ids2018_FalconID_test...
1,cse_cic_ids2018,RareGuard,0.993200,0.871994,0.855702,0.863215,0.992079,artifacts/models/cse_cic_ids2018_RareGuard.pt,artifacts/models/cse_cic_ids2018_RareGuard_tes...
6,edge_iiotset,FalconID,0.999400,0.999550,0.999100,0.999325,0.999400,artifacts/models/edge_iiotset_FalconID.pt,artifacts/models/edge_iiotset_FalconID_test_ve...
7,edge_iiotset,RareGuard,0.999400,0.999550,0.999100,0.999325,0.999400,artifacts/models/edge_iiotset_RareGuard.pt,artifacts/models/edge_iiotset_RareGuard_test_v...


In [4]:
# Optional: sync artifacts to the configured Google Drive folder.
# Keep --dry-run first if you want to preview.
!python scripts/sync_to_drive.py --dry-run
# !python scripts/sync_to_drive.py


rclone is not installed in this runtime.
Artifacts are already being written inside Google Drive because the project root is under /content/drive.
Current artifact folder: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION/artifacts
Skipping external rclone mirror. Continue with the next notebook.
